# The running balance that jumps ahead

**Track:** debug-the-ai | **Difficulty:** medium | **Tags:** window functions, running total, window frame

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("005-running-balance-tied-dates")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "transactions": """
UEFSMRUEFbAZFfYMTBWWAxUAEgAA2AwI38MABQEAsA0IANgNCABpDQgAuA0IAL0NCAQbxAUvBAANDQgAaw0YAIoNCACRDQgAjw0I
APsNCAC1DQgAtw0IAJgNCADhDQgAYA0IAKwNCABsDQgAcg0IAJcNCADgDQgA/Q0IAAwNiABVDRAABA0QAP4NEADvDQgAcw0IABoN
IAAYDQgA3Q0YAAsNEAC+DRAAhA0IAG0NCAASDSAAkw0QAFcNCAB/DQgAWQ0IAPMNCABhDQgA8g0IAF0NCAB5DQgAkg0IALwNCACk
DQgAXw0IAKoNCAAQDXgAhg0QAPYNCACeDQgA0A0IAA8NKADHDRAAtA0IAHwNCADGDQgAFA0oAKsNEAB0DQgAgw0IAFMNCAAXDSgA
5g0QAMgNCADpDQgAvw0IAF4NCAACDTAAgg0QANMNCAChDQgArg0IAK0NCACHDQgA+A0IAJwNCAB9DQgAiQ0IAAcNWADiDRAAlg0I
AOQNCADqDQgABg0oAFINEAAJDRAArw0QAO0NCAD0DQgACA0gAP8NEABuDQgA7A0IAJsNCACQDQgAbw0IANUNCACgDQgAAw1IAJ8N
EABbDQgAZw0IAGYNCADWDQgAyw0IABkNOAAVDQgAhQ0YABENEABwDRAAYw0IAIgNCACADQgAfg0IAHYNCABYDQgAAQ1AAHoNEABW
DQgA6w0IAMQNCAATDSgAew0QAMINCADUDQgA7g0IAMANCABoDQgAiw0IAJoNCACNDQgAYg0IAPUNCADbDQgAzQ0IAPwNCAD6DQgA
2Q0IAPENCABlDQgAtg0IAKcNCACUDQgAyQ0IAGQNCAB4DQgAuQ0IAHUNCABUDQgABQ3gANcNEADwDQgAwQ0IAGoNCAClDQgA6A0I
ANENCABxDQgAWg0IAM8NCAAODVgA9w0QAAANEACyDRAAyg0IAKINCABcDQgAug0IAIwNCACzDQgA4w0IAI4NCACBDQgAow0IAMUN
CACxDQgA3A0IAN4NCACmDQgAqQ0IAMwNCADaDQgAww0IAHcNCADODQgA+Q0IAOUNCADnDQgAnQ0IAAoN2ACZDRAAUQ0IAJUNCADS
DQgAqA0IPLvDAAAAAAAAFsQAAAAAAAAVABWyAxW6AywVlgMVEBUGFQYcGAgbxAAAAAAAABgIUcMAAAAAAAAWACgIG8QAAAAAAAAY
CFHDAAAAAAAAEREAAADZAfDYAwAAAJYDAQg1AAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4v
MDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6
e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vb6/wMHCw8TF
xsfIycoAAAAAABUEFZADFeIBTBUyFQASAADIAQjUAgAFAQDNDQgAvQ0IAMENCADRDQgAyQ0IAMINCADSDQgAyg0IANUNCADLDQgA
0w0IAL8NCADFDQgAyA0IAL4NCADHDQgA0A0IAM8NCADODQgAzA0IAMANCADDDQg8xgIAAAAAAADEAgAAAAAAABUAFZYCFZ4CLBWW
AxUQFQYVBhwYCNUCAAAAAAAAGAi9AgAAAAAAABYAKAjVAgAAAAAAABgIvQIAAAAAAAAREQAAAIsB8IoDAAAAlgMBBTUgADFIMYeg
pMhioY3i2HkwSgXRpIcyIgQS1YEGELEMNWjLFMNQKCQI0F2AsJlIuldEYQhAHChrLbgRagWKDJBOqZe+GO0ptNlzjaNCErMkGhVH
V6xJCd+pVDPANaPsJW9MVgyGxFRXrLVqRPvYSgvBSNM69Y1Gh60oBCUmu2Y6AAAAFQQVyAQV0gRMFZIBFQASAACkAvQjAbdOAACc
TgAAtU4AAIZOAACkTgAAqE4AANJOAADNTgAAjk4AAJFOAADDTgAAoE4AAKJOAACTTgAAgE4AAJtOAACHTgAAxk4AAHtOAADITgAA
v04AAMxOAACNTgAAz04AAHxOAACLTgAAfU4AAMBOAAB+TgAAp04AAJlOAAB/TgAAmk4AAMJOAACYTgAAsE4AAKxOAACfTgAAik4A
AKtOAADRTgAAjE4AAHpOAAC6TgAAvE4AAKlOAACzTgAAl04AAMpOAAC4TgAAuU4AAMtOAAC9TgAAg04AAK5OAACCTgAAiU4AAKpO
AADQTgAAvk4AAIVOAACWTgAAwU4AALZOAAChTgAArU4AAKVOAACeTgAApk4AAJBOAAC7TgAAlU4AAHlOAAAVABX+AhWGAywVlgMV
EBUGFQYcGATSTgAAGAR5TgAAFgAoBNJOAAAYBHlOAAAREQAAAL8B8L4DAAAAlgMBBzWAgGBAKBgOA0QioVgwGgDHYwBpACIHySRC
gRgMgErBMrgkMBnNxrHhQBIdzwd0sIREowOpZDqhQJCUCrUisVqcScolCjwsoZcJAYs1ZCyYahagbWaRAe2VGAhEExGnVhPYDCjL
YXBDpDIQTASHoeV0ppZg1+IhegmH7zeKKASbGxDUCN4gQogEJqC0Bh6WC6IZHUKRoU3EIRaHsUTKkxsCACDQ6JeDjBQro6hyRDZc
ICIKAAAAABUEFe4PFfgPTBWWAxUAEgAA9wf09gMAACYPKgAALWnI///ybB8AAAuxwAAAJuMUAAAWLrb///faW///+lbIAAAldBYA
ACdjAv//+C/n///x3F////iZw///8YwAAAAekfr///tq9P//7gks///50LP///0I6wAAKIdk///22n///++yLAAAF008///2MVv/
//HnTwAANPBYAAAXf27///BpxP//+SW0AAAEQtL//+qu3///7Rpn///xcuf//+22p///8YQX///q7kD///8IWAAAE8I0///vHJAA
ACGxyP//9bXrAAAQLTYAABdzUgAAHoO4///3zrwAAGMztP//9k/T///u3IAAAAQSMP//8dXUAABzpqQAABXw/gAAG8p0AAAhw1z/
/+u99///9zBv///zeuwAAAypmv//8bjTAAARcHQAAB46FgAALBrSAAAlsWr///nVYwAAC0s2///s9WQAAFRHyAAAHj3+///pICT/
//E3PP///jlQ///430v//+0lcAAALL7i///5xwj//+pQDAAADrFUAAAm8Cr///i3cwAAJag8////r2////gKZwAAIHl6///sESsA
ABXFPv//6Wiz///qn7z///qWvwAAFI1U///1TPAAAEl+2AAADM4gAAAMU0b///YTR///8/53///spCT///oUYAAAWBi8AAAS6ZoA
AEb02P//+NZo///yPaP///zP4wAAB63S///rhKQAAAmOrgAAJmWw///sKkT///mDdP//6dVk///rDWcAABZ9nv//8f+HAAAtRqAA
ABkLhgAALdDo///prGAAABBZWv//6200AABsW9j//+4pfwAAXsrMAAAhI8oAAGOBDAAAM7qUAAANxTIAACICcgAAGQ+gAABJXBQA
AA+Y+P//+j7b////E0j///Wz9wAAdOaQ///vcsv///yJ9wAAZVGAAAA5CZAAACpzYv//9fkbAAAYa/QAACZRxP//+DHbAAAeLWQA
AAxKGP//68r0///+s5QAABqkHgAAYhQ0///89qgAAA4F2P//+ry8AAAMKE7///kcVP//6lwPAAAZuVoAAA4I+AAAAlV2AAAjl4YA
ADnAYP//8K9M////hCwAAAjDKv//+nG8AABBe6wAABuT9v//7Ns4AAAr1FAAAAMSuP//8wEMAAAYcKQAACUsmv//6vX3AAACXg4A
AA2uJgAAAWAmAAAkxE7///eKR///+WrY///wKbQAAA04xP//9K8H///yx6AAAA0JgAAAAx4M///uFOT//+w6rAAAHfI2///rqN8A
AGa6DAAACe/y///qLOT///gr/wAAFSSyAAAmQ7T//+mKyP///4h4AAA8YKQAABO8Jv//9nTvAAAL/xj//+1UUP//92P/FQAVsgMV
ugMsFZYDFRAVBhUGHBgFAAB05pAYBf//6SAkFgAoBQAAdOaQGAX//+kgJBERAAAA2QHw2AMAAACWAwEINQABAgMEBQYHCAkKCwwN
Dg8QERITFBUWFxgZGhscHR4fICEiIyQlJicoKSorLC0uLzAxMjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldY
WVpbXF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp+goaKj
pKWmp6ipqqusra6vsLGys7S1tre4ubq7vL2+v8DBwsPExcbHyMnKAAAAAAAVBBlcNQAYBnNjaGVtYRUIABUEJQIYBnR4bl9pZAAV
BCUCGAphY2NvdW50X2lkABUCJQIYCHR4bl9kYXRlJQxMbAAAABUOFQoVAhgGYW1vdW50JQoVBBUULFwVBBUUAAAAFpYDGRwZTCYA
HBUEGTUABhAZGAZ0eG5faWQVAhaWAxaIHhbWESagDSYIHBgIG8QAAAAAAAAYCFHDAAAAAAAAFgAoCBvEAAAAAAAAGAhRwwAAAAAA
ABERABksFQQVABUCABUAFRAVAgA8KQYZJgCWAwAAACYAHBUEGTUABhAZGAphY2NvdW50X2lkFQIWlgMWygYWpAUm4BMm3hEcGAjV
AgAAAAAAABgIvQIAAAAAAAAWACgI1QIAAAAAAAAYCL0CAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAJYDAAAAJgAcFQIZ
NQAGEBkYCHR4bl9kYXRlFQIWlgMWzAgW3ggm9hsmghccGATSTgAAGAR5TgAAFgAoBNJOAAAYBHlOAAAREQAZLBUEFQAVAgAVABUQ
FQIAPCkGGSYAlgMAAAAmABwVDhk1AAYQGRgGYW1vdW50FQIWlgMWrhQWwBQm+i8m4B8cGAUAAHTmkBgF///pICQWACgFAAB05pAY
Bf//6SAkEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAJYDAAAAFsxBFpYDJggWmDQAGRwYDEFSUk9XOnNjaGVtYRiYAy8vLy8veWdC
QUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFRQUFBREFBQUFBZUFB
QUFFQUFBQUFFQUFBQVlQLy8vd0FBQVFjUUFBQUFJQUFBQUFRQUFBQUFBQUFBQmdBQUFHRnRiM1Z1ZEFBQUNBQU1BQVFBQ0FBSUFB
QUFDZ0FBQUFJQUFBQ1kvLy8vQUFBQkNCQUFBQUFnQUFBQUJBQUFBQUFBQUFBSUFBQUFkSGh1WDJSaGRHVUFBQVlBQ0FBR0FBWUFB
QUFBQUFBQXpQLy8vd0FBQVFJUUFBQUFIQUFBQUFRQUFBQUFBQUFBQ2dBQUFHRmpZMjkxYm5SZmFXUUFBTUQvLy84QUFBQUJRQUFB
QUJBQUZBQUlBQVlBQndBTUFBQUFFQUFRQUFBQUFBQUJBaEFBQUFBZ0FBQUFCQUFBQUFBQUFBQUdBQUFBZEhodVgybGtBQUFJQUF3
QUNBQUhBQWdBQUFBQUFBQUJRQUFBQUE9PQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGUwcAAAcAAAcAAAcAAAA
7gMAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "005-running-balance-tied-dates"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

transactions = spark.read.parquet((_data_dir / "transactions.parquet").resolve().as_posix())
transactions.createOrReplaceTempView("transactions")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'account_id': 'aac9b73d4b447851aefc0cce4ec8d10e47e3125eef4a23fdf025b10abe6f7b7b',
                         'amount': 'a8d14b90356c7fd7e7331b06e76b52b43a20bc91395f17873d411c5cee55e508',
                         'balance_after': '8c945bed430f2b05c5808361276d002d38df70d246fb0b928b6defd7942387df',
                         'txn_date': '2c9795fb4d9165a17f258800de58d119408948967b280be0b5c5527b1685204f',
                         'txn_id': 'df005c4a6dfce5c6bd19607882d92df42e13e2436d392e8f05a96fd46f42ed90'},
 'columns': [['account_id', 'bigint'],
             ['amount', 'decimal(10,2)'],
             ['balance_after', 'decimal(12,2)'],
             ['txn_date', 'date'],
             ['txn_id', 'bigint']],
 'fingerprint': '1655fb2c8a0532bf5a2f74bf278cf431687f99ec6a5de6d25a9103e42b0826d8',
 'order_matters': False,
 'row_count': 203}

check = make_check(EXPECTED)

## The situation

The statements team needs the account balance after every transaction. An AI assistant
wrote the query in the answer cell. It runs, returns one row per transaction, and for
many accounts every number is right.

Support has a complaint from a customer: two transactions on the same day both show
the same balance, and it is not the balance after either of them alone.

Find the bug and fix it.

## Input tables

### `transactions`

One row per transaction.

| Column | Type | Description |
|---|---|---|
| `txn_id` | BIGINT | Unique. Within a date, a lower `txn_id` was processed earlier. |
| `account_id` | BIGINT | The account. |
| `txn_date` | DATE | The date of the transaction. An account can have several on one date. |
| `amount` | DECIMAL(10,2) | Positive for money in, negative for money out. |

## Requirement

For every transaction, return the account's balance immediately after it.

- Every account starts at 0.
- Transactions of an account are applied in order of `txn_date`, and within a date
  in order of `txn_id`.
- `balance_after` is the sum of the account's amounts up to and including that
  transaction.

## Expected output

One row per transaction.

| Column | Type | Description |
|---|---|---|
| `txn_id` | BIGINT | |
| `account_id` | BIGINT | |
| `txn_date` | DATE | |
| `amount` | DECIMAL(10,2) | |
| `balance_after` | DECIMAL(12,2) | Balance right after this transaction. |

In [ ]:
transactions.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `transactions`.

In [ ]:
query = """
-- Running balance per account: cumulative sum of amounts in date order
SELECT
    txn_id,
    account_id,
    txn_date,
    amount,
    CAST(
        SUM(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date
        ) AS DECIMAL(12, 2)
    ) AS balance_after
FROM transactions
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)